In [2]:
import pandas as pd
from pathlib import Path


In [51]:
from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# LOAD DATA
# ============================================================

Proj_Folder = Path.cwd()

input_file = Proj_Folder / "raw" / "SV.csv"

df = pd.read_csv(
    input_file,
    low_memory=False
)

# Convert epoch to datetime
df["epoch"] = pd.to_datetime(
    df["epoch"],
    format="mixed",
    utc=True,
    errors="coerce"
)


# ============================================================
# BASIC DATASET INFORMATION
# ============================================================

print("=" * 70)
print("FULL DATASET")
print("=" * 70)

print("Rows:", len(df))

print(
    "Unique satellites:",
    df["idOnOrbit"].nunique()
)

print(
    "Date range:",
    df["epoch"].min(),
    "to",
    df["epoch"].max()
)

print("\nColumns:")
print(df.columns.tolist())

FULL DATASET
Rows: 3186577
Unique satellites: 2776
Date range: 2025-07-01 00:00:00+00:00 to 2025-07-31 23:59:59.816014+00:00

Columns:
['ypos', 'origNetwork', 'xvel', 'idOnOrbit', 'epoch', 'classificationMarking', 'source', 'createdAt', 'zpos', 'yvel', 'referenceFrame', 'idStateVector', 'satNo', 'origin', 'zvel', 'xpos', 'pedigree', 'dataMode', 'createdBy', 'descriptor', 'transactionId', 'tags', 'cov', 'covReferenceFrame', 'solarRadPressCoeff', 'dragCoeff', 'velUnc', 'posUnc', 'uct', 'origObjectId', 'dragModel', 'geopotentialModel', 'solarRadPress', 'inTrackThrust', 'solidEarthTides', 'lunarSolar', 'polarMotionY', 'polarMotionX', 'taiUtc', 'algorithm', 'cmOffset', 'solarFluxAPAvg', 'solarFluxF10', 'solarFluxF10Avg', 'leapSecondTime', 'stepSizeSelection', 'ut1Rate', 'thrustAccel', 'rms', 'sigmaVelUVW', 'integratorMode', 'bDot', 'stepMode', 'errorControl', 'revNo', 'edr', 'stepSize', 'partials', 'sigmaPosUVW', 'iau1980Terms', 'fixedStep', 'ut1Utc', 'eqCov', 'covMethod', 'recODSpan', 'act

In [52]:
# ============================================================
# SATELLITES BY ORBITAL REGIME
# ============================================================

regime_counts = (
    df.groupby("orbitRegime")["idOnOrbit"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="n_satellites")
)

print("\n")
print("=" * 70)
print("SATELLITES BY ORBITAL REGIME")
print("=" * 70)

print(regime_counts.to_string(index=False))



SATELLITES BY ORBITAL REGIME
orbitRegime  n_satellites
        LEO          1786
        GEO           668
        MEO           240
        HEO            53
        UNK            29


In [53]:
# ============================================================
# SATELLITES BY COUNTRY
# ============================================================

country_counts = (
    df.groupby("country")["idOnOrbit"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="n_satellites")
)

print("\n")
print("=" * 70)
print("SATELLITES BY COUNTRY")
print("=" * 70)

print(country_counts.to_string(index=False))



SATELLITES BY COUNTRY
country  n_satellites
    RUS          1095
    CHN          1064
    USA           274
    SES            50
    ESA            33
   ITSO            31
   EUTE            29
    JPN            28
    O3B            20
    CAN            20
    KOR            19
     IM            15
    GBR            12
    FRA            11
    FIN            10
    AUS             9
    BRA             9
    IRN             8
     AC             6
    DEU             6
    ISR             5
    ABS             4
    NOR             3
    ESP             3
    ITA             2
    QAT             2
    MEX             1
    ISS             1
    AZE             1
    PRK             1
   FRIT             1
    ARG             1
   TMMC             1
    CHL             1


In [54]:
# ============================================================
# COUNTRY x ORBITAL REGIME
# ============================================================

country_regime = (
    df.groupby(
        ["country", "orbitRegime"]
    )["idOnOrbit"]
    .nunique()
    .reset_index(
        name="n_satellites"
    )
)

country_regime_pivot = (
    country_regime
    .pivot(
        index="country",
        columns="orbitRegime",
        values="n_satellites"
    )
    .fillna(0)
    .astype(int)
)

# Add total unique counts across regimes for sorting
country_regime_pivot["Total"] = (
    country_regime_pivot.sum(axis=1)
)

country_regime_pivot = (
    country_regime_pivot
    .sort_values(
        "Total",
        ascending=False
    )
)

print("\n")
print("=" * 70)
print("SATELLITES BY COUNTRY AND ORBITAL REGIME")
print("=" * 70)

print(country_regime_pivot.to_string())



SATELLITES BY COUNTRY AND ORBITAL REGIME
orbitRegime  GEO  HEO  LEO  MEO  UNK  Total
country                                    
RUS          177   41  730  145    2   1095
CHN          148   10  867   39    0   1064
USA          126    0  117   31    0    274
SES           42    0    2    0    6     50
ESA            2    0    6   25    0     33
ITSO          31    0    0    0    0     31
EUTE          29    0    0    0    0     29
JPN           22    0    6    0    0     28
O3B            0    0    0    0   20     20
CAN           14    0    6    0    0     20
KOR            6    0   13    0    0     19
IM            14    1    0    0    0     15
GBR            9    0    3    0    0     12
FRA            5    0    6    0    0     11
FIN            0    0   10    0    0     10
AUS            8    0    1    0    0      9
BRA            7    0    2    0    0      9
IRN            0    0    8    0    0      8
AC             6    0    0    0    0      6
DEU            3    0    3    0  

In [55]:
# ============================================================
# GEO ONLY
# ============================================================

geo = df[
    df["orbitRegime"] == "GEO"
].copy()

print("\n")
print("=" * 70)
print("GEO DATASET")
print("=" * 70)

print(
    "Rows:",
    len(geo)
)

print(
    "Unique GEO satellites:",
    geo["idOnOrbit"].nunique()
)

print(
    "GEO date range:",
    geo["epoch"].min(),
    "to",
    geo["epoch"].max()
)



GEO DATASET
Rows: 2279591
Unique GEO satellites: 668
GEO date range: 2025-07-01 00:00:00+00:00 to 2025-07-31 23:59:59.816014+00:00


In [56]:
# ============================================================
# GEO SATELLITES BY COUNTRY
# ============================================================

geo_country_counts = (
    geo.groupby("country")["idOnOrbit"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(
        name="n_geo_satellites"
    )
)

geo_country_counts["percent_of_geo"] = (
    100
    * geo_country_counts["n_geo_satellites"]
    / geo["idOnOrbit"].nunique()
)

print("\n")
print("=" * 70)
print("GEO SATELLITES BY COUNTRY")
print("=" * 70)

print(
    geo_country_counts.to_string(
        index=False,
        formatters={
            "percent_of_geo":
                lambda x: f"{x:.2f}%"
        }
    )
)



GEO SATELLITES BY COUNTRY
country  n_geo_satellites percent_of_geo
    RUS               177         26.50%
    CHN               148         22.16%
    USA               126         18.86%
    SES                42          6.29%
   ITSO                31          4.64%
   EUTE                29          4.34%
    JPN                22          3.29%
    CAN                14          2.10%
     IM                14          2.10%
    GBR                 9          1.35%
    AUS                 8          1.20%
    BRA                 7          1.05%
    KOR                 6          0.90%
     AC                 6          0.90%
    FRA                 5          0.75%
    ISR                 4          0.60%
    ABS                 4          0.60%
    DEU                 3          0.45%
    NOR                 3          0.45%
    ESA                 2          0.30%
    QAT                 2          0.30%
    ESP                 1          0.15%
    MEX                 1    

In [60]:
# ============================================================
# USA GEO SATELLITES
# ============================================================

usa_geo = geo[
    geo["country"] == "USA"
].copy()

usa_satellites = (
    usa_geo[
        [
            "idOnOrbit",
            "commonName"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "commonName"
    )
)

print("\n")
print("=" * 70)
print("USA GEO SATELLITES")
print("=" * 70)

print(
    usa_satellites.to_string(
        index=False
    )
)

print(
    "\nTotal USA GEO satellites:",
    usa_geo["idOnOrbit"].nunique()
)



USA GEO SATELLITES
 idOnOrbit              commonName
     36868        AEHF 1 (USA 214)
     38254        AEHF 2 (USA 235)
     39256        AEHF 3 (USA 246)
     43651        AEHF 4 (USA 288)
     44481        AEHF 5 (USA 292)
     45465        AEHF 6 (USA 298)
     56371                ARCTURUS
     51287                  ASCENT
     26107                ASIASTAR
     23754              ECHOSTAR 1
     28935             ECHOSTAR 10
     33207             ECHOSTAR 11
     36499             ECHOSTAR 14
     36792             ECHOSTAR 15
     39008             ECHOSTAR 16
     38551             ECHOSTAR 17
     41592             ECHOSTAR 18
     41893             ECHOSTAR 19
     42749             ECHOSTAR 21
     42070             ECHOSTAR 23
     26038               GALAXY 11
     28790               GALAXY 14
     29236               GALAXY 16
     31307               GALAXY 17
     32951               GALAXY 18
     33376               GALAXY 19
     27854  GALAXY 23 (TELSTAR 13)